In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
file_path = 'Hotel Reservations.csv'
data = pd.read_csv(file_path)

In [3]:
data.head(20)

,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,arrival_year,arrival_month,arrival_date,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,2017,10,2,Offline,0,0,0,65.00,0,Not_Canceled
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,2018,11,6,Online,0,0,0,106.68,1,Not_Canceled
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,2018,2,28,Online,0,0,0,60.00,0,Canceled
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,2018,5,20,Online,0,0,0,100.00,0,Canceled
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,2018,4,11,Online,0,0,0,94.50,0,Canceled
5,INN00006,2,0,0,2,Meal Plan 2,0,Room_Type 1,346,2018,9,13,Online,0,0,0,115.00,1,Canceled
6,INN00007,2,0,1,3,Meal Plan 1,0,Room_Type 1,34,2017,10,15,Online,0,0,0,107.55,1,Not_Canceled
7,INN00008,2,0,1,3,Meal Plan 1,0,Room_Type 4,83,2018,12,26,Online,0,0,0,105.61,1,Not_Canceled
8,INN00009,3,0,0,4,Meal Plan 1,0,Room_Type 1,121,2018,7,6,Offline,0,0,0,96.90,1,Not_Canceled
9,INN00010,2,0,0,5,Meal Plan 1,0,Room_Type 4,44,2018,10,18,Online,0,0,0,133.44,3,Not_Canceled


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36275 entries, 0 to 36274
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Booking_ID                            36275 non-null  object 
 1   no_of_adults                          36275 non-null  int64  
 2   no_of_children                        36275 non-null  int64  
 3   no_of_weekend_nights                  36275 non-null  int64  
 4   no_of_week_nights                     36275 non-null  int64  
 5   type_of_meal_plan                     36275 non-null  object 
 6   required_car_parking_space            36275 non-null  int64  
 7   room_type_reserved                    36275 non-null  object 
 8   lead_time                             36275 non-null  int64  
 9   arrival_year                          36275 non-null  int64  
 10  arrival_month                         36275 non-null  int64  
 11  arrival_date   

In [5]:
data.isnull().sum()

#Great! There are no null values.

Booking_ID                              0
no_of_adults                            0
no_of_children                          0
no_of_weekend_nights                    0
no_of_week_nights                       0
type_of_meal_plan                       0
required_car_parking_space              0
room_type_reserved                      0
lead_time                               0
arrival_year                            0
arrival_month                           0
arrival_date                            0
market_segment_type                     0
repeated_guest                          0
no_of_previous_cancellations            0
no_of_previous_bookings_not_canceled    0
avg_price_per_room                      0
no_of_special_requests                  0
booking_status                          0
dtype: int64

In [6]:
data.select_dtypes(object)


,Booking_ID,type_of_meal_plan,room_type_reserved,market_segment_type,booking_status
0,INN00001,Meal Plan 1,Room_Type 1,Offline,Not_Canceled
1,INN00002,Not Selected,Room_Type 1,Online,Not_Canceled
2,INN00003,Meal Plan 1,Room_Type 1,Online,Canceled
3,INN00004,Meal Plan 1,Room_Type 1,Online,Canceled
4,INN00005,Not Selected,Room_Type 1,Online,Canceled
...,...,...,...,...,...
36270,INN36271,Meal Plan 1,Room_Type 4,Online,Not_Canceled
36271,INN36272,Meal Plan 1,Room_Type 1,Online,Canceled
36272,INN36273,Meal Plan 1,Room_Type 1,Online,Not_Canceled
36273,INN36274,Not Selected,Room_Type 1,Online,Canceled


In [7]:
# I'm curious about the categorical columns
categorical_columns = [
    'type_of_meal_plan', 
    'room_type_reserved', 
    'market_segment_type',
    'booking_status'
]

#...and their unique values
for col in categorical_columns:
    unique_values = data[col].unique()
    print(f"Unique values in '{col}':")
    print(unique_values)
    print()


Unique values in 'type_of_meal_plan':
['Meal Plan 1' 'Not Selected' 'Meal Plan 2' 'Meal Plan 3']

Unique values in 'room_type_reserved':
['Room_Type 1' 'Room_Type 4' 'Room_Type 2' 'Room_Type 6' 'Room_Type 5'
 'Room_Type 7' 'Room_Type 3']

Unique values in 'market_segment_type':
['Offline' 'Online' 'Corporate' 'Aviation' 'Complementary']

Unique values in 'booking_status':
['Not_Canceled' 'Canceled']



In [8]:
#I want to turn the arrival data into something useful for time. but I'm curious about one thing...

# Check for invalid date combinations
invalid_dates = []

for idx, row in data.iterrows():
    year = int(row['arrival_year'])
    month = int(row['arrival_month'])
    day = int(row['arrival_date'])
    
    try:
        pd.Timestamp(year=year, month=month, day=day)
    except ValueError:
        invalid_dates.append((idx, year, month, day))

# Display invalid date combinations
invalid_dates


[(2626, 2018, 2, 29),
 (3677, 2018, 2, 29),
 (5600, 2018, 2, 29),
 (6343, 2018, 2, 29),
 (7648, 2018, 2, 29),
 (8000, 2018, 2, 29),
 (8989, 2018, 2, 29),
 (9153, 2018, 2, 29),
 (9245, 2018, 2, 29),
 (9664, 2018, 2, 29),
 (9934, 2018, 2, 29),
 (10593, 2018, 2, 29),
 (10652, 2018, 2, 29),
 (10747, 2018, 2, 29),
 (11881, 2018, 2, 29),
 (13958, 2018, 2, 29),
 (14304, 2018, 2, 29),
 (15363, 2018, 2, 29),
 (15438, 2018, 2, 29),
 (17202, 2018, 2, 29),
 (18380, 2018, 2, 29),
 (18534, 2018, 2, 29),
 (18680, 2018, 2, 29),
 (19013, 2018, 2, 29),
 (20419, 2018, 2, 29),
 (21674, 2018, 2, 29),
 (21688, 2018, 2, 29),
 (26108, 2018, 2, 29),
 (27559, 2018, 2, 29),
 (27928, 2018, 2, 29),
 (30552, 2018, 2, 29),
 (30616, 2018, 2, 29),
 (30632, 2018, 2, 29),
 (30839, 2018, 2, 29),
 (32041, 2018, 2, 29),
 (34638, 2018, 2, 29),
 (35481, 2018, 2, 29)]

In [9]:
# Correct invalid dates by setting February 29 to February 28 for non-leap years
for idx, year, month, day in invalid_dates:
    if month == 2 and day == 29:
        data.at[idx, 'arrival_date'] = 28

# Combine arrival_year, arrival_month, and arrival_date into a single datetime column
data['date_of_arrival'] = pd.to_datetime(data[['arrival_year', 'arrival_month', 'arrival_date']].rename(columns={
    'arrival_year': 'year',
    'arrival_month': 'month',
    'arrival_date': 'day'
}))

# Display the updated DataFrame with the new arrival_date column
print("\nData with Combined Arrival Date:")
display(data[['arrival_year', 'arrival_month', 'arrival_date', 'date_of_arrival']].head())

# Optionally, drop the original arrival_year, arrival_month, and arrival_date columns
data.drop(columns=['arrival_year', 'arrival_month', 'arrival_date'], inplace=True)

# Display the DataFrame to ensure the columns are dropped
print("\nData After Dropping Original Arrival Columns:")
display(data.head())




Data with Combined Arrival Date:


,arrival_year,arrival_month,arrival_date,date_of_arrival
0,2017,10,2,2017-10-02
1,2018,11,6,2018-11-06
2,2018,2,28,2018-02-28
3,2018,5,20,2018-05-20
4,2018,4,11,2018-04-11



Data After Dropping Original Arrival Columns:


,Booking_ID,no_of_adults,no_of_children,no_of_weekend_nights,no_of_week_nights,type_of_meal_plan,required_car_parking_space,room_type_reserved,lead_time,market_segment_type,repeated_guest,no_of_previous_cancellations,no_of_previous_bookings_not_canceled,avg_price_per_room,no_of_special_requests,booking_status,date_of_arrival
0,INN00001,2,0,1,2,Meal Plan 1,0,Room_Type 1,224,Offline,0,0,0,65.00,0,Not_Canceled,2017-10-02
1,INN00002,2,0,2,3,Not Selected,0,Room_Type 1,5,Online,0,0,0,106.68,1,Not_Canceled,2018-11-06
2,INN00003,1,0,2,1,Meal Plan 1,0,Room_Type 1,1,Online,0,0,0,60.00,0,Canceled,2018-02-28
3,INN00004,2,0,0,2,Meal Plan 1,0,Room_Type 1,211,Online,0,0,0,100.00,0,Canceled,2018-05-20
4,INN00005,2,0,1,1,Not Selected,0,Room_Type 1,48,Online,0,0,0,94.50,0,Canceled,2018-04-11


## That should do for now!

In [10]:
# Save the cleaned dataset to a new CSV file
output_file_path = 'Cleaned_Hotel_Reservations.csv'
data.to_csv(output_file_path, index=False)

# Confirm the file is saved
print(f"Cleaned dataset saved to {output_file_path}")

Cleaned dataset saved to Cleaned_Hotel_Reservations.csv
